In [ ]:
import pandas as pd
import numpy as np
from extract import load_raw

df_raw = load_raw("cad", mode="latest")
df_past_raw = load_raw("fireball", mode="all")

cad = df_raw.copy()
fireball = df_past_raw.copy()

**Transformation: cad dataset**

In [3]:
cad.isnull().sum()
cad[cad["v_inf"].isna()]

,des,orbit_id,jd,cd,dist,dist_min,dist_max,v_rel,v_inf,t_sigma_f,h,diameter,diameter_sigma,fullname,_extracted_at,_api_version,_raw_file
719,2026 QQ8,1,2461162.408381293,2026-May-01 21:48,0.00772560278929482,0.00772321808213006,0.00772798762297442,0.795217418167035,NaN,00:06,31.367,NaN,NaN,(2026 QQ8),2026-11-01T02:53:31+00:00,1.5,20261101T025331640820Z.json
976,2026 QQ8,1,2461271.489274951,2026-Aug-18 23:45,0.00358999471838524,0.00358994476162161,0.00359004467761537,1.14605686661353,NaN,< 00:01,31.367,NaN,NaN,(2026 QQ8),2026-11-01T02:53:31+00:00,1.5,20261101T025331640820Z.json


In [4]:
cad['des'] = cad['des'].str.strip()
cad['jd'] = cad['jd'].astype("float")
cad["cd"] = pd.to_datetime(cad["cd"], format="%Y-%b-%d %H:%M", errors="raise")


In [5]:
cad['dist'] = cad['dist'].astype('float')
print(f"Distance in 'Au' (Astronomic unit):\n{cad['dist'].head()}")
cad['dist_km'] = cad['dist'] * 149597870.7
print(f"Distance in Km:\n{cad['dist_km'].head()}")
cad["dist_ld"] = cad["dist_km"] / 384400
print(f"Distance in Ld (Parameter for the distance between the Moon and the Earth):\n{cad['dist_ld'].head()}")


Distance in 'Au' (Astronomic unit):
0    0.019376
1    0.010783
2    0.020454
3    0.006225
4    0.019552
Name: dist, dtype: float64
Distance in Km:
0    2.898645e+06
1    1.613160e+06
2    3.059929e+06
3    9.311867e+05
4    2.924963e+06
Name: dist_km, dtype: float64
Distance in Ld (Parameter for the distance between the Moon and the Earth):
0    7.540699
1    4.196565
2    7.960274
3    2.422442
4    7.609165
Name: dist_ld, dtype: float64


In [6]:
for col in ["dist_min", "dist_max"]:
    cad[col] = pd.to_numeric(cad[col], errors="raise")
cad["dist_uncertainty_km"] = (cad["dist_max"] - cad["dist_min"]) * 149597870.7
cad["dist_uncertainty_pct"] = (cad["dist_max"] - cad["dist_min"]) / cad["dist"] * 100
ok = (cad["dist_min"] <= cad["dist"]) & (cad["dist"] <= cad["dist_max"])
print(ok.all())
print(f"2026 QQ8 Example\n{cad.loc[cad["des"] == "2026 QQ8", ["cd", "dist_ld", "dist_uncertainty_km", "dist_uncertainty_pct"]]}")


True
2026 QQ8 Example
                      cd   dist_ld  dist_uncertainty_km  dist_uncertainty_pct
719  2026-05-01 21:48:00  3.006591           713.513155              0.061737
976  2026-08-18 23:45:00  1.397127            14.947220              0.002783
1244 2026-12-13 13:41:00  3.516946           170.836061              0.012637


In [7]:
for col in ["v_rel", "v_inf"]:
    cad[col] = pd.to_numeric(cad[col], errors="raise")
cad["v_esc"] = np.sqrt(2 * 398600.4418 / cad["dist_km"])
cad["gravity_boost_kms"] = cad["v_rel"] - cad["v_inf"]
cad["gravitationally_bound"] = cad["v_rel"] < cad["v_esc"]
ok = np.isclose(cad["v_rel"]**2 - cad["v_inf"]**2, cad["v_esc"]**2, rtol=1e-2)
print(ok[cad["v_inf"].notna()].all())
exemplo = cad[cad["v_inf"].notna()].nsmallest(1, "dist")["des"].iloc[0]
print(f"{exemplo} Example\n{cad.loc[cad['des'] == exemplo, ['cd', 'v_rel', 'v_inf', 'gravity_boost_kms', 'gravitationally_bound']]}")
print(
    "v_rel: speed relative to Earth at closest approach, including Earth's gravity (km/s)\n"
    "v_inf: speed far from Earth, ignoring its gravity (km/s)\n"
    "gravity_boost_kms: extra speed gained from Earth's gravity (v_rel - v_inf)\n"
    "gravitationally_bound: True if the object was too slow to escape Earth's gravity"
)

True
2026 TP128 Example
                      cd      v_rel     v_inf  gravity_boost_kms  \
1199 2026-10-15 14:38:00  21.489969  18.85155           2.638419   

      gravitationally_bound  
1199                  False  
v_rel: speed relative to Earth at closest approach, including Earth's gravity (km/s)
v_inf: speed far from Earth, ignoring its gravity (km/s)
gravity_boost_kms: extra speed gained from Earth's gravity (v_rel - v_inf)
gravitationally_bound: True if the object was too slow to escape Earth's gravity


In [8]:
s = cad["t_sigma_f"].str.strip()
cad["t_sigma_lt_1min"] = s.str.startswith("<")
parts = s.str.removeprefix("< ").str.extract(r"^(?:(\d+)_)?(\d{2}):(\d{2})$")
if parts[[1, 2]].isna().any().any():
    raise ValueError("Inexpected format in t_sigma_f")
parts = parts.fillna("0").astype(int)
cad["t_sigma_min"] = parts[0] * 1440 + parts[1] * 60 + parts[2]

In [9]:
for col in ['h', 'diameter']:
    cad[col] = pd.to_numeric(cad[col], errors="raise")
cad["diameter_est_min_m"] = 1329 / np.sqrt(0.25) * 10 ** (-cad["h"] / 5) * 1000
cad["diameter_est_max_m"] = 1329 / np.sqrt(0.05) * 10 ** (-cad["h"] / 5) * 1000
cad["has_measured_diameter"] = cad["diameter"].notna()

In [10]:
cad['fullname'] = cad['fullname'].str.strip()
regex_nome_proprio = r"^(?:\d+\s+)?[A-Za-z][a-z\-]+(?:\s+[A-Za-z][a-z\-]+)*\s*\(.+\)$"
contem_nome_proprio = cad['fullname'].str.match(regex_nome_proprio, na=False)
asteroides_famosos = cad.loc[contem_nome_proprio, 'fullname'].drop_duplicates()
if not asteroides_famosos.empty:
    print(f"Asteroides com nome próprio encontrados próximos à Terra:")
    print(asteroides_famosos.to_string(index=False))
else:
    print("Nenhum asteroide com nome próprio nominal próximo da Terra no momento.")

Nenhum asteroide com nome próprio nominal próximo da Terra no momento.


In [11]:
cad['_extracted_at'] = pd.to_datetime(cad["_extracted_at"], errors="raise")

**Transformation: Fireball dataset**

In [12]:
print(fireball.columns)

Index(['date', 'energy', 'impact-e', 'lat', 'lat-dir', 'lon', 'lon-dir', 'alt',
       'vel', '_extracted_at', '_api_version', '_raw_file'],
      dtype='str')


In [13]:
fireball['date'] = pd.to_datetime(fireball['date'], errors='raise')
fireball['lat'] = fireball['lat'].astype("float")
fireball['lat'] = fireball['lat'] * np.where(fireball['lat-dir'] == 'S', -1, 1)
print(fireball['lat'])

0      -41.8
1      -37.6
2       -0.2
3       54.4
4      -19.3
        ... 
1070    51.3
1071    78.7
1072     7.5
1073    -4.1
1074   -41.8
Name: lat, Length: 1075, dtype: float64


In [14]:
fireball['lon'] = fireball['lon'].astype("float")
fireball['lon-dir'] = fireball['lon-dir'].str.strip()
fireball['lon'] = fireball['lon'] * np.where(fireball['lon-dir'] == 'W', -1, 1)
print(fireball['lon'])

0      -173.1
1      -161.6
2      -129.6
3      -100.1
4       -28.1
        ...  
1070   -100.9
1071      6.3
1072    142.8
1073    124.3
1074   -173.1
Name: lon, Length: 1075, dtype: float64


In [19]:
antes = len(fireball)
fireball = fireball.drop_duplicates(subset=["date", "lat", "lon"], keep="last").reset_index(drop=True)
print(f"Duplicates removed: {antes - len(fireball)}")

Duplicates removed: 0


In [15]:
for col in ["alt", "vel", "energy", "impact-e"]:
    fireball[col] = pd.to_numeric(fireball[col], errors="raise")
fireball['energy_joules'] = fireball['energy'] * (10**10)
fireball['impact_energy_joules'] = fireball['impact-e'] * (4.185 * (10**12))
nulos_alt = fireball['alt'].isnull().sum()
print(nulos_alt)

472


In [16]:
fireball['_extracted_at'] = pd.to_datetime(fireball["_extracted_at"], errors="raise")
print(fireball['_extracted_at'].head())

0   2026-11-01 02:45:03+00:00
1   2026-11-01 02:45:03+00:00
2   2026-11-01 02:45:03+00:00
3   2026-11-01 02:45:03+00:00
4   2026-11-01 02:45:03+00:00
Name: _extracted_at, dtype: datetime64[us, UTC]


In [17]:
fireball = fireball.rename(columns=lambda c: c.replace("-", "_"))